# Central Tendency Comparison — Reusable Template (`HousingCT_Py`)

Drop in any numeric column (or several groups) and compute mean / median / mode, a skew diagnosis, a histogram overlay, an audience blurb, and a two-knob simulation.

Copy this notebook, point `PATHS` at your CSVs, set `VALUE_COL` and `GROUP_COL`.



In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from collections import Counter

# ---- configure ----
PATHS = {
    "Brooklyn": "data/brooklyn-one-bed.csv",
    "Manhattan": "data/manhattan-one-bed.csv",
    "Queens": "data/queens-one-bed.csv",
}
VALUE_COL = "rent"
GROUP_COL = None          # e.g. "borough" if everything is already in one file
FILTERS = {
    # "Brooklyn": ("bedrooms", 1),   # uncomment for a fair 1-bed card
}


In [ ]:
def load_groups(paths, value_col, filters=None):
    out = {}
    for name, path in paths.items():
        df = pd.read_csv(path)
        if filters and name in filters:
            col, val = filters[name]
            df = df.loc[df[col] == val]
        out[name] = df[value_col].dropna().astype(float)
    return out

def central(x):
    x = pd.Series(x).dropna()
    mode_res = stats.mode(x, keepdims=True)
    return {
        "n": int(len(x)),
        "mean": float(x.mean()),
        "median": float(x.median()),
        "mode": float(mode_res.mode[0]),
        "mode_count": int(mode_res.count[0]),
        "q1": float(x.quantile(0.25)),
        "q3": float(x.quantile(0.75)),
        "iqr": float(x.quantile(0.75) - x.quantile(0.25)),
        "min": float(x.min()),
        "max": float(x.max()),
        "skew_mean_minus_median": float(x.mean() - x.median()),
        "pct_above_mean": float((x > x.mean()).mean()),
    }

groups = load_groups(PATHS, VALUE_COL, FILTERS)
table = pd.DataFrame({k: central(v) for k, v in groups.items()}).T
table


## Histogram overlay


In [ ]:
fig, axes = plt.subplots(1, len(groups), figsize=(4.2 * len(groups), 4), squeeze=False)
for ax, (name, x) in zip(axes[0], groups.items()):
    s = central(x)
    xmax = np.percentile(x, 99)
    ax.hist(x[x <= xmax], bins=20, edgecolor="white", alpha=0.85)
    ax.axvline(s["mean"], color="black", label=f"mean {s['mean']:.0f}")
    ax.axvline(s["median"], color="orange", ls="--", label=f"median {s['median']:.0f}")
    ax.axvline(s["mode"], color="purple", ls=":", label=f"mode {s['mode']:.0f}")
    ax.set_title(f"{name} n={s['n']}")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## Simulation knobs


In [ ]:
SEED = 42
N_REP = 300
N_SAMPLE = 150
N_OUTLIERS = 4
OUTLIER_VALUE = None      # default = 3 * max of first group

rng = np.random.default_rng(SEED)
first = next(iter(groups.values())).to_numpy()
if OUTLIER_VALUE is None:
    OUTLIER_VALUE = float(np.max(first) * 2)

boot_mean = [rng.choice(first, N_SAMPLE, True).mean() for _ in range(N_REP)]
boot_med = [np.median(rng.choice(first, N_SAMPLE, True)) for _ in range(N_REP)]
print("bootstrap mean width", np.percentile(boot_mean, 97.5) - np.percentile(boot_mean, 2.5))
print("bootstrap median width", np.percentile(boot_med, 97.5) - np.percentile(boot_med, 2.5))

sample = rng.choice(first, 400, True)
inj = np.concatenate([sample, np.full(N_OUTLIERS, OUTLIER_VALUE)])
print("outlier Δ mean", inj.mean() - sample.mean())
print("outlier Δ median", np.median(inj) - np.median(sample))


## Audience stub

Fill one sentence per row after you read `table`.

- Analyst:
- Technician:
- Executive:
- Nonspecialist:

